In [ ]:
# !pip install -q huggingface_hub
!hf auth login
# HFAKwA6TlbqTwmFQbRf51Ejva3HV5pj

? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: X3YF-ZWVN

    Waiting for authorization...
Token is valid.
The token `oauth-Prabhas2002` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-Prabhas2002`
Note: This token will be refreshed automatically when it expires.


In [ ]:
# !pip install -q datasets

from datasets import load_dataset

ds = load_dataset("MRR24/English_to_Telugu_Bilingual_Sentence_Pairs")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['Input', 'Output', 'Instruction', 'Conversation'],
        num_rows: 155798
    })
})


In [ ]:
from datasets import DatasetDict

# 1. Drop unused columns
ds = ds.remove_columns(["Instruction", "Conversation"])

# 2. Split: 90% train, 10% temp -> temp split in half (5% valid, 5% test)
split = ds["train"].train_test_split(test_size=0.05, seed=42)

ds = DatasetDict({
    "train": split["train"],
    "test": split["test"],
})

print(ds)
for k, v in ds.items():
    print(k, len(v))

DatasetDict({
    train: Dataset({
        features: ['Input', 'Output'],
        num_rows: 148008
    })
    test: Dataset({
        features: ['Input', 'Output'],
        num_rows: 7790
    })
})
train 148008
test 7790


In [ ]:
for i in range(5):
    print(ds["train"][i])

{'Input': 'This book belongs to Tom.', 'Output': 'ఈ పుస్తకం టామ్\u200cకు చెందినది.'}
{'Input': "I don't know when Tom will leave Japan.", 'Output': 'టామ్ ఎప్పుడు జపాన్ నుండి బయలుదేరుతాడో నాకు తెలియదు.'}
{'Input': 'Hey, get up.', 'Output': 'హే, లేచి.'}
{'Input': 'Tom promised to help Mary with her homework.', 'Output': 'టామ్ మేరీకి హోంవర్క్ తో సహాయం చేస్తానని వాగ్దానం చేశాడు.'}
{'Input': 'Tom never said he wanted to go with us.', 'Output': 'టామ్ ఎప్పుడూ మాతో వెళ్లాలని అనుకోలేదు.'}


In [ ]:
import re, unicodedata

# Set this to match your data
TGT_RANGE = ("\u0C00", "\u0C7F")   # Telugu. For Hindi use ("\u0900", "\u097F")

def norm(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", s)).strip()

def script_ratio(s, lo, hi):
    letters = [c for c in s if c.isalpha()]
    return sum(lo <= c <= hi for c in letters) / len(letters) if letters else 0

def keep(ex):
    s, t = ex["src"], ex["tgt"]
    if not s or not t: return False
    ns, nt = len(s.split()), len(t.split())
    if ns < 2 or nt < 2 or ns > 80 or nt > 80: return False
    if not (0.4 <= ns / nt <= 2.5): return False
    if script_ratio(t, *TGT_RANGE) < 0.7: return False     # target must be in the right script
    if script_ratio(s, *TGT_RANGE) > 0.1: return False     # source must be English
    return True

ds = ds.map(lambda ex: {"src": norm(ex["Input"]), "tgt": norm(ex["Output"])})
ds = ds.filter(keep)
print({k: len(v) for k, v in ds.items()})

Map:   0%|          | 0/148008 [00:00<?, ? examples/s]

Map:   0%|          | 0/7790 [00:00<?, ? examples/s]

Filter:   0%|          | 0/148008 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7790 [00:00<?, ? examples/s]

{'train': 146457, 'test': 7707}


In [ ]:
for i in range(5):
    print(ds["train"][i])

{'Input': 'This book belongs to Tom.', 'Output': 'ఈ పుస్తకం టామ్\u200cకు చెందినది.', 'src': 'This book belongs to Tom.', 'tgt': 'ఈ పుస్తకం టామ్\u200cకు చెందినది.'}
{'Input': "I don't know when Tom will leave Japan.", 'Output': 'టామ్ ఎప్పుడు జపాన్ నుండి బయలుదేరుతాడో నాకు తెలియదు.', 'src': "I don't know when Tom will leave Japan.", 'tgt': 'టామ్ ఎప్పుడు జపాన్ నుండి బయలుదేరుతాడో నాకు తెలియదు.'}
{'Input': 'Hey, get up.', 'Output': 'హే, లేచి.', 'src': 'Hey, get up.', 'tgt': 'హే, లేచి.'}
{'Input': 'Tom promised to help Mary with her homework.', 'Output': 'టామ్ మేరీకి హోంవర్క్ తో సహాయం చేస్తానని వాగ్దానం చేశాడు.', 'src': 'Tom promised to help Mary with her homework.', 'tgt': 'టామ్ మేరీకి హోంవర్క్ తో సహాయం చేస్తానని వాగ్దానం చేశాడు.'}
{'Input': 'Tom never said he wanted to go with us.', 'Output': 'టామ్ ఎప్పుడూ మాతో వెళ్లాలని అనుకోలేదు.', 'src': 'Tom never said he wanted to go with us.', 'tgt': 'టామ్ ఎప్పుడూ మాతో వెళ్లాలని అనుకోలేదు.'}


In [ ]:
import os
os.makedirs("data", exist_ok=True)

with open("data/train.Input", "w") as fs, open("data/train.Output", "w") as ft:
    for ex in ds["train"]:
        fs.write(ex["Input"] + "\n")
        ft.write(ex["Output"] + "\n")

In [ ]:
import sentencepiece as spm
import os
os.makedirs("tokenizer", exist_ok=True)

for name in ["Input", "Output"]:
    spm.SentencePieceTrainer.train(
        input=f"data/train.{name}",
        model_prefix=f"tokenizer/spm_{name}",
        vocab_size=4000,            # ~140k pairs is small, so 8k to 16k is enough
        model_type="bpe",
        character_coverage=1.0,     # keeps rare Indic conjuncts
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    )

sp_src = spm.SentencePieceProcessor(model_file="tokenizer/spm_Input.model")
sp_tgt = spm.SentencePieceProcessor(model_file="tokenizer/spm_Output.model")

### Re-Tokenizing with Explicit BOS/EOS Tokens

We will now update the tokenization process to ensure:
* **Source (English)**: Appends `<EOS>` (ID 3) so the encoder knows where the sentence ends before padding.
* **Target (Telugu)**: Prepends `<BOS>` (ID 2) and appends `<EOS>` (ID 3) so the decoder has a starting trigger and a stopping signal.

In [ ]:
def tokenize_fn_updated(examples):
    # Source: Append EOS (3)
    src_ids = [sp_src.encode(text) + [3] for text in examples["src"]]

    # Target: Prepend BOS (2) and Append EOS (3)
    tgt_ids = [[2] + sp_tgt.encode(text) + [3] for text in examples["tgt"]]

    return {
        "input_ids": src_ids,
        "labels": tgt_ids
    }

# Map the updated tokenization function over our train and test datasets
tokenized_ds = ds.map(
    tokenize_fn_updated,
    batched=True,
    remove_columns=ds["train"].column_names
)

print("Updated Tokenization complete!")
print("Sample tokenized input:", tokenized_ds["train"][0]["input_ids"])
print("Sample tokenized label:", tokenized_ds["train"][0]["labels"])

Map:   0%|          | 0/146457 [00:00<?, ? examples/s]

Map:   0%|          | 0/7707 [00:00<?, ? examples/s]

Updated Tokenization complete!
Sample tokenized input: [211, 376, 3011, 17, 13, 3925, 3]
Sample tokenized label: [2, 117, 678, 14, 126, 3651, 3878, 3]


In [ ]:
print(sp_src.encode('Hi there'))
print(sp_src.decode([75, 3921, 286]))

print(sp_tgt.encode('ఎలా ఉన్నావు'))
# Use sp_tgt here to decode Telugu token IDs back to Telugu text:
print(sp_tgt.decode([281, 102, 169]))

[75, 3920, 286]
Hh there
[283, 101, 170]
న్నారా చాలాడ్


In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    # Extract input_ids and labels from batch dictionary
    input_ids = [torch.tensor(item["input_ids"]) for item in batch]
    labels = [torch.tensor(item["labels"]) for item in batch]

    # Dynamically pad sequences in the batch to match the maximum length sequence
    # batch_first=True makes the tensor shape (batch_size, sequence_length)
    # padding_value=0 as specified by pad_id=0 in SentencePiece
    padded_input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=0)

    return {
        "input_ids": padded_input_ids,
        "labels": padded_labels
    }

# Create PyTorch DataLoaders
batch_size = 32

train_loader = DataLoader(
    tokenized_ds["train"],
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    tokenized_ds["test"],
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

# Quick test to verify shapes and tokens
batch = next(iter(train_loader))
print("Input (Source) batch shape:", batch["input_ids"].shape)
print("Labels (Target) batch shape:", batch["labels"].shape)
print("\nSample padded Input (Source) tensor (no BOS, ends with 3 + pad 0):\n", batch["input_ids"][0])
print("\nSample padded Labels (Target) tensor (starts with BOS 2, ends with 3 + pad 0):\n", batch["labels"][0])

Input (Source) batch shape: torch.Size([32, 14])
Labels (Target) batch shape: torch.Size([32, 16])

Sample padded Input (Source) tensor (no BOS, ends with 3 + pad 0):
 tensor([ 139, 3934, 3919,  199, 1218,   17, 1634,  657,   25, 3925,    3,    0,
           0,    0])

Sample padded Labels (Target) tensor (starts with BOS 2, ends with 3 + pad 0):
 tensor([   2, 2192,  407, 1422,  153,  102, 2748,  143, 3878,    3,    0,    0,
           0,    0,    0,    0])


In [ ]:
# # Rnn to Rnn seq model.
# import random
# import torch
# import torch.nn as nn

# class RNNSeq(nn.Module):
#   def __init__(self, input_size, hidden_size, vocab_size):
#     super().__init__()
#     self.hidden_size = hidden_size
#     self.vocab_size = vocab_size
#     self.proj1  = nn.Embedding(vocab_size, input_size, padding_idx=0)
#     self.proj2  = nn.Embedding(vocab_size, input_size, padding_idx=0)
#     self.Rnn1  = nn.RNN(input_size, hidden_size, batch_first=True)
#     self.Rnn2  = nn.RNN(input_size, hidden_size, batch_first=True)
#     self.linear = nn.Linear(hidden_size, vocab_size)

#   def forward(self, x, y=None, teacher_forcing_ratio=0.5):
#     # x: (B, src_len) -> Source inputs
#     # y: (B, tgt_len) -> Target labels (used for teacher forcing during training)
#     batch_size = x.size(0)

#     # 1. ENCODER
#     x_emb = self.proj1(x)                                 # (B, src_len, input_size)
#     _, h_enc = self.Rnn1(x_emb)                           # h_enc: (1, B, hidden_size)

#     # If targets are not provided (e.g., inference), we will generate autoregressively without teacher forcing
#     if y is None:
#         teacher_forcing_ratio = 0.0
#         max_len = 50
#     else:
#         max_len = y.size(1)

#     # Initialize decoder hidden state with encoder's final hidden state
#     h_dec = h_enc

#     # Start token is BOS (2) for every sequence in the batch
#     dec_input = torch.full((batch_size, 1), 2, dtype=torch.long, device=x.device)

#     outputs = []
#     for t in range(1, max_len):
#         # Embed the decoder input: (B, 1) -> (B, 1, input_size)
#         dec_emb = self.proj2(dec_input)

#         # Decode one step: output is (B, 1, hidden_size)
#         dec_out, h_dec = self.Rnn2(dec_emb, h_dec)

#         # Predict vocabulary logits: (B, vocab_size)
#         logits = self.linear(dec_out.squeeze(1))
#         outputs.append(logits.unsqueeze(1))

#         # Get the highest-probability token ID
#         top1 = logits.argmax(1, keepdim=True) # (B, 1)

#         # Decide whether to use teacher forcing or model's own predictions
#         use_teacher_forcing = random.random() < teacher_forcing_ratio
#         if use_teacher_forcing and y is not None:
#             dec_input = y[:, t].unsqueeze(1)  # ground truth
#         else:
#             dec_input = top1                  # predicted token

#     # Stack output logits over time dimension -> (B, max_len - 1, vocab_size)
#     return torch.cat(outputs, dim=1)


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = RNNSeq(input_size=256, hidden_size=512, vocab_size=4000).to(device)
# print(model)

# #test model with rand tokens
# rand_input = torch.randint(0, 4000, (32, 11)).to(device)
# rand_output = model(rand_input)
# print(rand_output.shape)

### Upgraded Architecture: GRUSeq with Regularization
This is our improved Sequence-to-Sequence model using Gated Recurrent Units (`nn.GRU`), embedding dropout, recurrent dropout, and weight decay to combat overfitting and dramatically improve validation convergence.

In [ ]:
# import random
# import torch
# import torch.nn as nn
# import torch.optim as optim

# class GRUSeq(nn.Module):
#     def __init__(self, input_size, hidden_size, vocab_size, dropout=0.2):
#         super().__init__()
#         self.hidden_size = hidden_size
#         self.vocab_size = vocab_size

#         # Embedding layers with dropout
#         self.proj1 = nn.Embedding(vocab_size, input_size, padding_idx=0)
#         self.proj2 = nn.Embedding(vocab_size, input_size, padding_idx=0)
#         self.dropout = nn.Dropout(dropout)

#         # 2-layer GRU with recurrent dropout
#         self.Rnn1 = nn.GRU(input_size, hidden_size, batch_first=True, num_layers=2, dropout=dropout if dropout > 0 else 0)
#         self.Rnn2 = nn.GRU(input_size, hidden_size, batch_first=True, num_layers=2, dropout=dropout if dropout > 0 else 0)

#         self.linear = nn.Linear(hidden_size, vocab_size)

#     def forward(self, x, y=None, teacher_forcing_ratio=0.5):
#         batch_size = x.size(0)

#         # Encoder
#         x_emb = self.dropout(self.proj1(x))
#         _, h_enc = self.Rnn1(x_emb) # GRU hidden state: (num_layers, B, hidden_size)

#         if y is None:
#             teacher_forcing_ratio = 0.0
#             max_len = 50
#         else:
#             max_len = y.size(1)

#         h_dec = h_enc
#         dec_input = torch.full((batch_size, 1), 2, dtype=torch.long, device=x.device)

#         outputs = []
#         for t in range(1, max_len):
#             dec_emb = self.dropout(self.proj2(dec_input))
#             dec_out, h_dec = self.Rnn2(dec_emb, h_dec)

#             logits = self.linear(dec_out.squeeze(1))
#             outputs.append(logits.unsqueeze(1))

#             top1 = logits.argmax(1, keepdim=True)

#             use_teacher_forcing = random.random() < teacher_forcing_ratio
#             if use_teacher_forcing and y is not None:
#                 dec_input = y[:, t].unsqueeze(1)
#             else:
#                 dec_input = top1

#         return torch.cat(outputs, dim=1)

# # Initialize the upgraded GRU model
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = GRUSeq(input_size=256, hidden_size=512, vocab_size=4000, dropout=0.2).to(device)
# print(model)

# # Configure Optimizer with Weight Decay (L2 regularization)
# optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [ ]:
# import torch.optim as optim
# from tqdm.auto import tqdm

# # Hyperparameters
# epochs = 5
# learning_rate = 1e-3

# # Loss and Optimizer
# criterion = nn.CrossEntropyLoss(ignore_index=0)
# optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# def train_epoch(model, loader, optimizer, criterion, device):
#     model.train()
#     epoch_loss = 0

#     for batch in tqdm(loader, desc="Training", leave=False):
#         src = batch["input_ids"].to(device)
#         tgt = batch["labels"].to(device)

#         optimizer.zero_grad()

#         # Forward pass: model yields shapes (B, tgt_len - 1, vocab_size)
#         output = model(src, tgt, teacher_forcing_ratio=0.5)

#         # Align targets: we ignore the BOS token (index 0) in targets
#         # target shape: (B, tgt_len - 1)
#         targets = tgt[:, 1:]

#         # Reshape to compute cross-entropy
#         # output: (B * (tgt_len - 1), vocab_size)
#         # targets: (B * (tgt_len - 1))
#         loss = criterion(output.reshape(-1, output.size(-1)), targets.reshape(-1))

#         loss.backward()
#         # Clip gradients to prevent exploding gradients in RNNs
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

#         optimizer.step()

#         epoch_loss += loss.item()

#     return epoch_loss / len(loader)

# def evaluate_epoch(model, loader, criterion, device):
#     model.eval()
#     epoch_loss = 0

#     with torch.no_grad():
#         for batch in tqdm(loader, desc="Evaluating", leave=False):
#             src = batch["input_ids"].to(device)
#             tgt = batch["labels"].to(device)

#             # Set teacher_forcing_ratio=0 during validation to see real performance
#             output = model(src, tgt, teacher_forcing_ratio=0.0)
#             targets = tgt[:, 1:]

#             loss = criterion(output.reshape(-1, output.size(-1)), targets.reshape(-1))
#             epoch_loss += loss.item()

#     return epoch_loss / len(loader)

### Fixing DataLoader Desynchronization & Re-Training
We will recreate the dataloaders to use the corrected `tokenized_ds` dataset (built in cell `3361193a`), initialize a fresh model, and run training to ensure proper convergence with correct BOS/EOS alignment.

In [ ]:
# for epoch in range(1, epochs + 1):
#     train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
#     val_loss = evaluate_epoch(model, test_loader, criterion, device)
#     print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Test Loss: {val_loss:.4f}")

### Inference Module (Translation Decoder)
Now we'll write an inference helper function `translate_sentence` that:
1. Tokenizes an input English text.
2. Appends the `<EOS>` token (`3`) to match our source-token training protocol.
3. Feeds it into the Encoder to generate the context hidden state.
4. Uses greedy decoding step-by-step to predict Telugu tokens, halting immediately when the model outputs `<EOS>` (`3`).

In [ ]:
# def translate_sentence(model, sentence, sp_src, sp_tgt, device, max_len=50):
#     model.eval()

#     # 1. Tokenize the source sentence and append EOS (3)
#     src_tokens = sp_src.encode(sentence) + [3]
#     src_tensor = torch.tensor(src_tokens, dtype=torch.long, device=device).unsqueeze(0) # (1, src_len)

#     with torch.no_grad():
#         # 2. Encode source sequence
#         x_emb = model.proj1(src_tensor)
#         _, h_enc = model.Rnn1(x_emb)

#         h_dec = h_enc
#         # Seed the decoder with BOS (2)
#         dec_input = torch.full((1, 1), 2, dtype=torch.long, device=device)

#         decoded_tokens = []

#         # 3. Autoregressive decoding
#         for _ in range(max_len):
#             dec_emb = model.proj2(dec_input)
#             dec_out, h_dec = model.Rnn2(dec_emb, h_dec)

#             logits = model.linear(dec_out.squeeze(1)) # (1, vocab_size)
#             top_token = logits.argmax(1).item()

#             # If model predicts EOS (3), translation is complete!
#             if top_token == 3:
#                 break

#             decoded_tokens.append(top_token)
#             # Use model's own output as next input token
#             dec_input = torch.tensor([[top_token]], dtype=torch.long, device=device)

#     # Decode output IDs back to native Telugu text
#     translation = sp_tgt.decode(decoded_tokens)
#     return translation

### Test translation on original sample requested

In [ ]:
# # Test translation on sample sentence
# sample_input = "Hi there, how are you doing"
# output_translation = translate_sentence(model, sample_input, sp_src, sp_tgt, device)
# print(f"Source: {sample_input}")
# print(f"Telugu Translation: {output_translation}")

In [ ]:
#TRAIN TRANSFORMER

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

PAD, UNK, BOS, EOS = 0, 1, 2, 3

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        split = lambda x: x.view(B, -1, self.h, self.dk).transpose(1, 2)  # (B,h,L,dk)
        q, k, v = split(self.wq(q)), split(self.wk(k)), split(self.wv(v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)             # (B,h,Lq,Lk)
        if mask is not None:                                              # mask: True = keep
            scores = scores.masked_fill(~mask, -1e9)
        attn = self.drop(F.softmax(scores, dim=-1))
        out = (attn @ v).transpose(1, 2).contiguous().view(B, -1, self.h * self.dk)
        return self.wo(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x): return self.net(x)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return self.drop(x + self.pe[:, :x.size(1)])

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.n1, self.n2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, src_mask):
        h = self.n1(x)
        x = x + self.drop(self.attn(h, h, h, src_mask))
        return x + self.drop(self.ff(self.n2(x)))

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.n1, self.n2, self.n3 = (nn.LayerNorm(d_model) for _ in range(3))
        self.drop = nn.Dropout(dropout)
    def forward(self, x, enc, src_mask, tgt_mask):
        h = self.n1(x)
        x = x + self.drop(self.self_attn(h, h, h, tgt_mask))
        x = x + self.drop(self.cross_attn(self.n2(x), enc, enc, src_mask))
        return x + self.drop(self.ff(self.n3(x)))

class TransformerSeq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, n_heads=4,
                 n_layers=3, d_ff=1024, dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=PAD)
        self.pos = PositionalEncoding(d_model, max_len, dropout)
        self.enc = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dec = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.enc_norm, self.dec_norm = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, tgt_vocab)
        self.out.weight = self.tgt_emb.weight          # weight tying (target side)
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    # ---- masks (True = attend) ----
    def src_mask(self, x):
        return (x != PAD).unsqueeze(1).unsqueeze(2)                     # (B,1,1,S)

    def tgt_mask(self, y):
        T = y.size(1)
        pad = (y != PAD).unsqueeze(1).unsqueeze(2)                      # (B,1,1,T)
        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=y.device))
        return pad & causal                                             # (B,1,T,T)

    def encode(self, x, src_mask):
        h = self.pos(self.src_emb(x) * math.sqrt(self.d_model))
        for layer in self.enc: h = layer(h, src_mask)
        return self.enc_norm(h)

    def decode(self, y, enc, src_mask):
        h = self.pos(self.tgt_emb(y) * math.sqrt(self.d_model))
        tm = self.tgt_mask(y)
        for layer in self.dec: h = layer(h, enc, src_mask, tm)
        return self.out(self.dec_norm(h))

    def forward(self, x, y):
        """x: (B,S) source ids, y: (B,T) target ids starting with BOS.
        Returns logits (B,T-1,V) that predict y[:,1:] (same as the GRU)."""
        sm = self.src_mask(x)
        return self.decode(y[:, :-1], self.encode(x, sm), sm)


# ---- init ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SRC_VOCAB, TGT_VOCAB = 4000 , 4000     # use sp_src.get_piece_size(), sp_tgt.get_piece_size()
model = TransformerSeq(SRC_VOCAB, TGT_VOCAB, d_model=256, n_heads=8,
                       n_layers=6, d_ff=1024, dropout=0.2).to(device)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# ---- optimizer, warmup schedule, loss ----
optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
warmup = 4000
scheduler = optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: model.d_model ** -0.5 * min((s + 1) ** -0.5, (s + 1) * warmup ** -1.5))
criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)

params: 13.1M


In [ ]:
# write traininig loop
import torch.optim as optim
from tqdm.auto import tqdm

In [ ]:
def train_epoch_transformer(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    epoch_loss = 0

    for batch in tqdm(loader, desc="Training Transformer", leave=False):
        src = batch["input_ids"].to(device)  # (B, S)
        tgt = batch["labels"].to(device)     # (B, T)

        optimizer.zero_grad()

        # Forward pass: transformer expects (src, tgt)
        # internally it slices target to y[:, :-1] to predict y[:, 1:]
        output = model(src, tgt)             # (B, T-1, vocab_size)

        targets = tgt[:, 1:]                 # (B, T-1)

        loss = criterion(output.reshape(-1, output.size(-1)), targets.reshape(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()  # Step learning rate scheduler

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

def evaluate_epoch_transformer(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating Transformer", leave=False):
            src = batch["input_ids"].to(device)
            tgt = batch["labels"].to(device)

            output = model(src, tgt)
            targets = tgt[:, 1:]

            loss = criterion(output.reshape(-1, output.size(-1)), targets.reshape(-1))
            epoch_loss += loss.item()

    return epoch_loss / len(loader)

# Run training
epochs = 5
for epoch in range(1, epochs + 1):
    train_loss = train_epoch_transformer(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss = evaluate_epoch_transformer(model, test_loader, criterion, device)
    print(f"[Transformer] Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Test Loss: {val_loss:.4f}")

Training Transformer:   0%|          | 0/4577 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
#save and load trained model
import os

os.makedirs("models", exist_ok=True)
save_path = "models/transformer_translation.pt"

# Save model weights
torch.save(model.state_dict(), save_path)
print(f"Model saved successfully to {save_path}!")

# Code to load back the weights:
loaded_model = TransformerSeq(SRC_VOCAB, TGT_VOCAB, d_model=256, n_heads=8, n_layers=6, d_ff=1024, dropout=0.2).to(device)
loaded_model.load_state_dict(torch.load(save_path, map_location=device))
loaded_model.eval()

In [ ]:
def translate_sentence_transformer(model, sentence, sp_src, sp_tgt, device, max_len=50):
    model.eval()

    # 1. Tokenize source and append EOS (3)
    src_tokens = sp_src.encode(sentence) + [3]
    src_tensor = torch.tensor(src_tokens, dtype=torch.long, device=device).unsqueeze(0) # (1, src_len)

    with torch.no_grad():
        # Encode source sequence once
        src_mask = model.src_mask(src_tensor)
        memory = model.encode(src_tensor, src_mask)

        # Seed target generation with BOS (2)
        ys = torch.full((1, 1), 2, dtype=torch.long, device=device)

        decoded_tokens = []
        for _ in range(max_len):
            # Decode current output tokens
            out = model.decode(ys, memory, src_mask)
            prob = out[:, -1]
            next_word = prob.argmax(dim=-1).item()

            if next_word == 3:  # Stop decoding when we predict EOS (3)
                break

            decoded_tokens.append(next_word)
            ys = torch.cat([ys, torch.tensor([[next_word]], dtype=torch.long, device=device)], dim=1)

    translation = sp_tgt.decode(decoded_tokens)
    return translation

# Test translation using our trained Transformer model
sample_input_transformer = "how are you"
output_translation_transformer = translate_sentence_transformer(loaded_model, sample_input_transformer, sp_src, sp_tgt, device)
print(f"Source: {sample_input_transformer}")
print(f"Transformer Translation: {output_translation_transformer}")

Source: how are you
Transformer Translation: మీరు మీరు మీరు?
